# DEM-Based Flow Analysis: Elevation-Aware Dam Accessibility

## Purpose
Determine which CPIS in arid SSA can plausibly receive gravity-fed surface water from
nearby dams, using elevation profiles from an SRTM-derived DEM rather than relying
solely on radial (straight-line) distance.

## Narrative context
This notebook is **Step 3** in the water source attribution arc and the **core
methodological contribution** of this research:

1. *CPIS expansion* — Notebooks in `Code/1_analyze_data`
2. *NDWI activity* — `5_ndwi_analysis.ipynb`
3. **Elevation-aware dam accessibility ← This notebook**
   The targeting ratio analysis in `Code/1_analyze_data/3_Dams_AEI_Targeting_Ratios.ipynb`
   showed weak distance-concentration patterns. This notebook explains why: a CPIS 10 km
   from a dam but 200 m uphill cannot receive gravity-fed irrigation water. By comparing
   CPIS and dam elevations from the existing SRTM DEM and using RichDEM to compute D8
   flow direction, we define a 'serviceable' dam as one where the CPIS lies
   topographically downslope within a feasible distance. This is what makes the
   '>50 km from a serviceable dam' finding credible.
4. *Groundwater* — `7_groundwater_gp_regression.ipynb`
5. *Anomalies* — `8_anomaly_detection.ipynb`

## Inputs
| Dataset | Config key | Notes |
|---------|-----------|-------|
| DEM (SRTM-derived, reprojected) | `Africa_Elevation_Reprojected_tif_path` | In repo |
| GDW dams (arid SSA) | `GDW_Arid_SSA_Final_shp_path` | In repo |
| CPIS polygons (arid SSA) | `SSA_Combined_CPIS_All_shp_path` | In repo |
| Africa boundaries | `Africa_boundaries_shp_path` | For visualization |
| Arid SSA regions | `SSA_All_by_Country_shp_path` | For visualization |

## Outputs
| File | Config key | Description |
|------|-----------|-------------|
| `CPIS_Elevation_Classified.shp` | `CPIS_Elevation_Classified_shp_path` | CPIS labeled feasible / infeasible_elevation / infeasible_distance |
| `Dam_CPIS_Elevation_Profiles.csv` | `Dam_CPIS_Profiles_csv_path` | Pairwise dam–CPIS elevation statistics |

In [ ]:
# --- Import Required Libraries and Utilities ---
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import Affine
import richdem as rd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Code.utils.utility import load_config, resolve_path

config = load_config()

## Load DEM, Dams, and CPIS Data

In [ ]:
# --- Load spatial data ---
dem_path = resolve_path(config['Africa_Elevation_Reprojected_tif_path'])

with rasterio.open(dem_path) as src:
    dem_meta = src.meta.copy()
    dem_crs = src.crs
    dem_nodata = src.nodata
    print(f"DEM loaded: {dem_meta['width']} x {dem_meta['height']} pixels")
    print(f"CRS: {dem_crs}")
    print(f"Bounds: {src.bounds}")

# GDW dams
dams = gpd.read_file(resolve_path(config['GDW_Arid_SSA_Final_shp_path']))
dams = dams.to_crs(dem_crs)
print(f"\nDams loaded: {len(dams)}")

# CPIS polygons (compute centroids for point-based elevation sampling)
cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))
cpis = cpis.to_crs(dem_crs)
cpis['centroid_x'] = cpis.geometry.centroid.x
cpis['centroid_y'] = cpis.geometry.centroid.y
print(f"CPIS loaded: {len(cpis)}")

# Boundaries for visualization
arid_ssa = gpd.read_file(resolve_path(config['SSA_All_by_Country_shp_path'])).to_crs(dem_crs)
arid_ssa['geometry'] = arid_ssa['geometry'].simplify(500)
af_bnds = gpd.read_file(resolve_path(config['Africa_boundaries_shp_path'])).to_crs(dem_crs)
af_bnds['geometry'] = af_bnds['geometry'].simplify(500)

## Extract Elevation at CPIS Centroids and Dam Locations

In [ ]:
# --- Sample DEM elevation at point locations ---
def sample_elevation(dem_path, coords_xy, nodata_val):
    """Sample DEM at (x, y) coordinate pairs. Returns array of elevation values."""
    with rasterio.open(dem_path) as src:
        raw = list(src.sample(coords_xy))
    elevs = np.array([v[0] for v in raw], dtype=float)
    if nodata_val is not None:
        elevs[elevs == nodata_val] = np.nan
    return elevs

# CPIS centroids
cpis_coords = list(zip(cpis['centroid_x'], cpis['centroid_y']))
cpis['elev_m'] = sample_elevation(dem_path, cpis_coords, dem_nodata)
print(f"CPIS elevation — mean: {np.nanmean(cpis['elev_m']):.0f} m, "
      f"range: [{np.nanmin(cpis['elev_m']):.0f}, {np.nanmax(cpis['elev_m']):.0f}] m")

# Dam locations
dam_coords = list(zip(dams.geometry.x, dams.geometry.y))
dams['elev_m'] = sample_elevation(dem_path, dam_coords, dem_nodata)
print(f"Dam elevation  — mean: {np.nanmean(dams['elev_m']):.0f} m, "
      f"range: [{np.nanmin(dams['elev_m']):.0f}, {np.nanmax(dams['elev_m']):.0f}] m")

## Pairwise Dam–CPIS Elevation Analysis

For each CPIS, we find its nearest dam and compute:
- **Straight-line distance** (m)
- **Elevation difference** = dam_elev − cpis_elev
  - Positive → CPIS is *below* the dam (gravity-fed irrigation physically feasible)
  - Negative → CPIS is *above* the dam (gravity-fed irrigation infeasible)

Classification:
- `feasible` — CPIS is downhill from nearest dam AND within 50 km
- `infeasible_distance` — nearest dam is > 50 km away
- `infeasible_elevation` — CPIS is uphill from nearest dam despite being within 50 km

In [ ]:
# --- Nearest-dam distance and elevation difference ---
DISTANCE_THRESHOLD_M = 50_000  # 50 km

dam_xy = np.column_stack([dams.geometry.x, dams.geometry.y])
cpis_xy = np.column_stack([cpis['centroid_x'], cpis['centroid_y']])

tree = cKDTree(dam_xy)
distances, indices = tree.query(cpis_xy, k=1)

cpis = cpis.copy()
cpis['nearest_dam_dist_m'] = distances
cpis['nearest_dam_elev_m'] = dams.iloc[indices]['elev_m'].values
cpis['elev_diff_m'] = cpis['nearest_dam_elev_m'] - cpis['elev_m']  # positive = CPIS below dam

def classify_cpis(row):
    if row['nearest_dam_dist_m'] > DISTANCE_THRESHOLD_M:
        return 'infeasible_distance'
    elif pd.isna(row['elev_diff_m']):
        return 'unknown'
    elif row['elev_diff_m'] >= 0:
        return 'feasible'
    else:
        return 'infeasible_elevation'

cpis['accessibility'] = cpis.apply(classify_cpis, axis=1)

counts = cpis['accessibility'].value_counts()
total = len(cpis)
print("CPIS Accessibility Classification:")
print("=" * 48)
for cat, n in counts.items():
    print(f"  {cat:<35} {n:5d} ({100*n/total:.1f}%)")
print(f"  {'TOTAL':<35} {total:5d}")

## Flow Direction Analysis with RichDEM

RichDEM computes D8 flow direction from a depression-filled DEM, identifying the
downstream drainage path from every pixel. Processing the full Africa DEM is
memory-intensive; this cell operates on a subsampled version for demonstration.
Set `SAMPLE_BBOX` to restrict to a specific region of interest.

In [ ]:
# --- D8 flow direction on a subsampled DEM ---
# Set SAMPLE_BBOX = (xmin, ymin, xmax, ymax) in DEM CRS units to restrict to a subregion.
# None uses the full DEM at 1/4 resolution for a quick overview.
SAMPLE_BBOX = None

with rasterio.open(dem_path) as src:
    if SAMPLE_BBOX:
        from rasterio.windows import from_bounds as win_from_bounds
        window = win_from_bounds(*SAMPLE_BBOX, transform=src.transform)
        dem_array = src.read(1, window=window).astype('float64')
        sample_transform = src.window_transform(window)
    else:
        # Read at 1/4 resolution for a quick overview
        out_h, out_w = src.height // 4, src.width // 4
        dem_array = src.read(1, out_shape=(out_h, out_w)).astype('float64')
        scale_x = src.width / out_w
        scale_y = src.height / out_h
        sample_transform = src.transform * Affine.scale(scale_x, scale_y)
    nodata_local = src.nodata

if nodata_local is not None:
    dem_array[dem_array == nodata_local] = np.nan

# Fill depressions and compute flow direction
dem_rd = rd.rdarray(dem_array, no_data=np.nan)
rd.FillDepressions(dem_rd, in_place=True)
flow_dir = rd.FlowDirD8(dem_rd)

print(f"DEM shape (subsampled): {dem_array.shape}")
print(f"Elevation range: [{np.nanmin(dem_array):.0f}, {np.nanmax(dem_array):.0f}] m")
print("D8 flow direction computed.")

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
plt.subplots_adjust(wspace=0.1, left=0.05, right=0.95, top=0.9, bottom=0.05)

im0 = axes[0].imshow(
    dem_array, cmap='terrain', origin='upper',
    vmin=np.nanpercentile(dem_array, 2), vmax=np.nanpercentile(dem_array, 98)
)
axes[0].set_title('DEM (SRTM-derived, subsampled)', fontsize=18)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)', fraction=0.04)

im1 = axes[1].imshow(flow_dir, cmap='tab10', origin='upper')
axes[1].set_title('D8 Flow Direction', fontsize=18)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], label='Flow direction code (1–128)', fraction=0.04)

fig.suptitle('DEM and Derived D8 Flow Direction — Arid SSA Subset',
             fontsize=22, fontweight='bold', y=0.97)
plt.show()

## Compare Elevation-Aware vs Radial Distance Accessibility

At each distance threshold, what fraction of CPIS that appear 'nearby' a dam
are actually uphill and therefore unreachable by gravity-fed irrigation?

This comparison directly explains why the radial targeting ratios in notebook 3
showed weak spatial concentration at close range.

In [ ]:
# --- Distance threshold comparison ---
thresholds = [10_000, 20_000, 30_000, 50_000, 75_000, 100_000]

rows = []
for thresh in thresholds:
    within = cpis['nearest_dam_dist_m'] <= thresh
    n_within = within.sum()
    n_feasible = ((cpis['nearest_dam_dist_m'] <= thresh) &
                  (cpis['accessibility'] == 'feasible')).sum()
    n_uphill = ((cpis['nearest_dam_dist_m'] <= thresh) &
                (cpis['accessibility'] == 'infeasible_elevation')).sum()
    rows.append({
        'Distance threshold (km)': thresh / 1000,
        'CPIS within distance (%)': round(100 * n_within / len(cpis), 1),
        'Elevation-feasible (%)': round(100 * n_feasible / len(cpis), 1),
        'Uphill of dam (% of nearby)': round(100 * n_uphill / n_within, 1) if n_within > 0 else 0
    })

comp_df = pd.DataFrame(rows)
print("Radial proximity vs elevation-feasible proximity:")
print(comp_df.to_string(index=False))

# Save elevation profiles
profiles_path = resolve_path(config['Dam_CPIS_Profiles_csv_path'])
os.makedirs(os.path.dirname(profiles_path), exist_ok=True)
cpis[['centroid_x', 'centroid_y', 'elev_m',
      'nearest_dam_dist_m', 'nearest_dam_elev_m',
      'elev_diff_m', 'accessibility']].to_csv(profiles_path)
print(f"\nDam-CPIS profiles saved: {profiles_path}")

## Visualize Accessibility Classification

CPIS color-coded by elevation-based accessibility:
- **Green:** Below nearest dam and within 50 km — gravity-fed water feasible
- **Orange:** Uphill from nearest dam despite being within 50 km — infeasible by elevation
- **Red:** Nearest dam > 50 km away — infeasible by distance

In [ ]:
# --- Map accessibility classification ---
color_map = {
    'feasible': '#1a9850',
    'infeasible_elevation': '#f46d43',
    'infeasible_distance': '#d73027',
    'unknown': '#cccccc'
}

fig, ax = plt.subplots(1, 1, figsize=(18, 14))

non_arid = af_bnds.overlay(arid_ssa, how='difference', keep_geom_type=False)
non_arid.plot(ax=ax, color='lightgray', hatch='//', edgecolor='black',
              linewidth=0.4, alpha=0.6)
arid_ssa.boundary.plot(ax=ax, color='black', linewidth=1.0, linestyle='--')
dams.plot(ax=ax, color='navy', markersize=6, alpha=0.5, label='Dams (GDW)')

for cat, color in color_map.items():
    subset = cpis[cpis['accessibility'] == cat]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, markersize=5, alpha=0.8,
                    label=f"{cat.replace('_', ' ').title()} ({len(subset):,})")

ax.set_title(
    f'CPIS Accessibility Classification (elevation-aware, {DISTANCE_THRESHOLD_M/1000:.0f} km threshold)',
    fontsize=20, fontweight='bold'
)
ax.set_axis_off()
ax.legend(fontsize=13, loc='lower left', framealpha=0.9)
plt.tight_layout()
plt.show()

# Save classified shapefile
out_path = resolve_path(config['CPIS_Elevation_Classified_shp_path'])
os.makedirs(os.path.dirname(out_path), exist_ok=True)
cpis_out = cpis.drop(columns=['centroid_x', 'centroid_y']).copy()
cpis_out.to_file(out_path)
print(f"Classified CPIS saved: {out_path}")